In [1]:
import pandas as pd
import re

In [2]:
head_labels = 'Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,UniqueCarrier,AirlineID,Carrier,TailNum,FlightNum,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime'.split(',')
columnNameToLabel = {}
for i in range(len(head_labels)):
    columnNameToLabel['column' + str(i+1)] = head_labels[i]

In [3]:
def lineToDataStruct(lineElements):

    index_dependant = 0
    for element in lineElements:
        if element.startswith('"dependant":'):
            break
        index_dependant += 1

    rhs = lineElements[index_dependant + 1].split(':')[1]
    for removeChar in ['"','[',']','{','}']:
        rhs = rhs.replace(removeChar, '')
    determinant_index = 0
    rhs = columnNameToLabel[rhs]

    for element in lineElements:
        if element.startswith('"determinant'):
            #print('found at ' + str(determinant_index))
            break
        determinant_index += 1

    X = [] # RHS
    for s in lineElements[determinant_index + 1: index_dependant]:
        match = re.search(r'"columnIdentifier":"(column\d+)"', s)
        if match:
            result = match.group(1)
            X.append(columnNameToLabel[result])

    return X,rhs

In [4]:
data = []
with open('2025-11-14_17-00-10_fds.csv', 'r') as file:
    for line in file:
        lhs, rhs = lineToDataStruct(line.strip().split(","))
        prettyPrint = lambda lst: ', '.join(lst)
        #print(prettyPrint(lhs), '->',rhs)
        data .append( (lhs, rhs) )

In [5]:
count = 0
for tuple in data:
    if all(str != 'WheelsOn' for str in tuple[0]):
        count+=1

In [9]:
import random
ids = random.sample(range(len(data)), 100)

In [11]:
sample = [data[i] for i in ids]

In [15]:
sample[2]

(['OriginCityName',
  'CRSArrTime',
  'ArrTime',
  'ActualElapsedTime',
  'AirTime',
  'Distance'],
 'CRSElapsedTime')

In [22]:
def find_fds_for_rhs(target_rhs, data,sorted=False):
    results = []
    for tuple in data:
        lhs, rhs = tuple
        if rhs ==target_rhs :
            results.append(tuple)
    if sorted:
        results.sort(key=lambda x: len(x[0]))
    return results

In [25]:
genuine_fds = [find_fds_for_rhs('ArrDelay', data, sorted = True)[0]]

In [27]:
firsts = [find_fds_for_rhs(label, data, sorted = True)[0] for label in head_labels]

In [ ]:
firsts

[(['ArrTime'], 'Year'),
 (['ArrTime'], 'Quarter'),
 (['ArrTime'], 'Month'),
 (['FlightDate'], 'DayofMonth'),
 (['DayofMonth'], 'DayOfWeek'),
 (['DayofMonth'], 'FlightDate'),
 (['ArrTime'], 'UniqueCarrier'),
 (['ArrTime'], 'AirlineID'),
 (['ArrTime'], 'Carrier'),
 (['FlightNum', 'CRSArrTime', 'DayofMonth'], 'TailNum'),
 (['CRSDepTime', 'CRSArrTime', 'DayofMonth'], 'FlightNum'),
 (['OriginAirportSeqID'], 'OriginAirportID'),
 (['OriginAirportID'], 'OriginAirportSeqID'),
 (['OriginAirportID'], 'OriginCityMarketID'),
 (['OriginAirportID'], 'Origin'),
 (['OriginAirportID'], 'OriginCityName'),
 (['OriginAirportID'], 'OriginState'),
 (['OriginAirportID'], 'OriginStateFips'),
 (['OriginAirportID'], 'OriginStateName'),
 (['OriginAirportID'], 'OriginWac'),
 (['DestAirportSeqID'], 'DestAirportID'),
 (['DestAirportID'], 'DestAirportSeqID'),
 (['DestAirportID'], 'DestCityMarketID'),
 (['DestAirportID'], 'Dest'),
 (['DestAirportID'], 'DestCityName'),
 (['DestAirportID'], 'DestState'),
 (['DestAirport

# Handlabeling test data

## Based on different sampling methods we construct a data set with both genuine and non genuine fd 